In python we have "Dynamic typing" instead of static typing. This makes it easy to learn as a beginner but can be a problem for industry grade products cuz, 
- There is no default type validation   
    Type validation checks if the input data is of correct data type   
    ex-  age: int
- There is no defualt data validation    
    Data validation checks if the input data is correct event if it has correct data type    
    ex- age: int, cannot be negative, does it have realistic range etc 


In [2]:
#Example (manualy) 
def insert_info(name: str, age: int):
    if type(name)==str and type(age)==int:
        if age<0:
            raise ValueError("Invalid value")
        print(name)
        print(age)
        print("Info inserted")
    
    else:
        raise TypeError("Invalid data type")


insert_info("Arun",19)

Arun
19
Info inserted


But this brute method is not scalable, there can be better ways to do the same thing in a big code space

### How Pydantic works: 
1. Define a pydantic model (class) that inherits from "BaseModel" in pydantic
    - This defines ideal schema of data
    - Which includes expected fields, datatypes, constraints 

2. Instantiate the model with raw input data for validation
    - Pydantic will automatically check the data and convert it in expected format if possible
    - Else, raise error 

3. Pass the validated object into functions 

In [3]:
from pydantic import BaseModel

#define pydantic model (class)
class Patient(BaseModel):
    name: str
    age: int

#Instantiate the model with raw input data
patient1_info = {"name":"Arun", "age":"19"}   # raw input data in form of dict 
patient1 = Patient(**patient1_info)         # "**" opens the dictionary

#Pass the validated model object into functions 
def insert_info(patient: Patient):
    print(patient.name)
    print(patient.age)
    print("Info inserted ")

insert_info(patient1)

Arun
19
Info inserted 


In [4]:
#Using different dtypes and Optional 
from pydantic import BaseModel
from typing import List,Dict, Optional


class Patient(BaseModel):
    name: str
    age: int
    weight: float
    married: Optional[bool] = 'No'     #This field is optional to be in instance. Defualt value is 'No'
    allergies: Optional[List[str]] = None
    additional_info: Dict[str,str]

patient1_info = {"name":"Arun", 
                 "age":"19", 
                 "weight":56.4, 
                 "allergies":['pollen','peanuts'], 
                 "additional_info":{'Last_fever':"jan 2026", 'BP':'150'}
                 }
patient1 = Patient(**patient1_info)

def insert_info(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.married)
    print(patient.allergies)
    print(patient.additional_info)
    print("Info inserted ")

insert_info(patient1)

Arun
19
56.4
No
['pollen', 'peanuts']
{'Last_fever': 'jan 2026', 'BP': '150'}
Info inserted 


We could have used 'list' and 'dict' instead of 'List' and 'Dict', but that would only validated if the field is list/dict or not,  
If we want to validate data inside those list/dicts we have to import them from `typing` module

_There are also some custom dtypes in pydantic_

In [5]:
#Example 
from pydantic import BaseModel, AnyUrl, EmailStr
from typing import List,Dict, Optional


class Patient(BaseModel):
    name: str
    age: int
    linkedin_url: AnyUrl   #Validate url
    email: EmailStr        #Validate email addresses
    married: Optional[bool] = 'No'     
    allergies: Optional[List[str]] = None
    additional_info: Optional[Dict[str,str]] = None

patient1_info = {"name":"Arun", 
                 "age":"19", 
                 "linkedin_url": "https://www.linkedin.com/in/arunnegi-tech/",
                 "email": "arunnegi1112@gmail.com"}
patient1 = Patient(**patient1_info)

def insert_info(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.linkedin_url)
    print(patient.email)
    print(patient.married)
    print(patient.allergies)
    print(patient.additional_info)
    print("Info inserted ")

insert_info(patient1)

Arun
19
https://www.linkedin.com/in/arunnegi-tech/
arunnegi1112@gmail.com
No
None
None
Info inserted 


### Data validation 
- For custom data validation or add some metadata, we use "Field"  
Field is used to add constraints and metadata to fields in data 

In [6]:
from pydantic import Field

class Patient(BaseModel):
    name: str = Field(max_length=50)
    age: int = Field(ge=0, lt=120)
    linkedin_url: AnyUrl  
    email: EmailStr        
    married: Optional[bool] = 'No'     
    allergies: Optional[List[str]] = None
    additional_info: Optional[Dict[str,str]] = None

patient1_info = {"name":"Arun", 
                 "age":"19", 
                 "linkedin_url": "https://www.linkedin.com/in/arunnegi-tech/",
                 "email": "arunnegi1112@gmail.com"}
patient1 = Patient(**patient1_info)

def insert_info(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.linkedin_url)
    print(patient.email)
    print(patient.married)
    print(patient.allergies)
    print(patient.additional_info)
    print("Info inserted ")

insert_info(patient1)

Arun
19
https://www.linkedin.com/in/arunnegi-tech/
arunnegi1112@gmail.com
No
None
None
Info inserted 


- To add metadata we use "typing.Annotated"   
Annotated is used to add metadata to a dtype in python. Python ignores this metadata but FastAPI and Pydantic still uses it   
Basically _Annotated exists to attach metadata to a type without changing the type._

syntax:
```python
Field: Annotated[dtype, Field()]

In [7]:
from typing import Annotated

class Patient(BaseModel):
    name: Annotated[str, Field(max_length=50, title='Name of the patient', description="Give name of the patient in less than 50 characters", examples=['Arun','Tiklu'])]
    age: int = Field(ge=0, lt=120)
    linkedin_url: AnyUrl  

patient1_info = {"name":"Arun", 
                 "age":"19", 
                 "linkedin_url": "https://www.linkedin.com/in/arunnegi-tech/",
                 "email": "arunnegi1112@gmail.com"}
patient1 = Patient(**patient1_info)

def insert_info(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.linkedin_url)

    print("Info inserted ")

insert_info(patient1)

Arun
19
https://www.linkedin.com/in/arunnegi-tech/
Info inserted 


Also note that, pydantic coerce the data to correct dtype (if its possible), but if we dont want that we can just simply
```python 
age: int = Field(ge=0, lt=120, strict = True)

#### Field Validator
If we want to add some more constraints in our fields, then we use this field validator   

For example, consider the above Patient class, lets assume that all the patients are of either hdfc bank or icici bank, Then to validate this we will use field validator on 'email'

In [ ]:
from pydantic import field_validator

class Patient(BaseModel):
    name: str
    age: int
    email: EmailStr

    #Now we want to keep only hdfc, icici emails
    #Add this decorator and pass required field, and then create a validation function
    @field_validator('email')
    @classmethod
    def check_mail(cls, value):     #cls =model class, value= field value  
        valid_domains= ['hdfc.com', 'icici.com']
        domain = value.split('@')[-1]
        if domain not in valid_domains:
            raise ValueError('Invalid Email')
        
        return value




patient1_info = {'name':'Arun', 'age':19, 'email':'arunnegi1112@hdfc.com'}
patient1 = Patient(**patient1_info)

def insert_info(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.email)
    
    print('Info inserted')

insert_info(patient1)

Arun
19
arunnegi1112@hdfc.com
Info inserted


### Model validator
If there is need of more than one fields for validation then this is used.   

ex: Lets add a validation that, if someones age is more than 60 then it is must to have a emergency contact number 

In [14]:
from pydantic import model_validator
from typing import Self
class Patient(BaseModel):
    name: str
    age: int
    contact_details: Dict[str,str]

    @model_validator(mode='after')
    def validate_emergency_contact(self)-> Self:  #cls: class, model: whole pydantic model

        if self.age>=60 and 'emergency' not in self.contact_details:
            raise ValueError("Add an emergency number for the patient if their age is more than equal to 60")
        
        return self
    


patient1_info = {'name':'Arun', 'age':19, 'contact_details':{'Dad':'23455','emergency':'24444'}}
patient1 = Patient(**patient1_info)



def insert_info(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.contact_details)
    
    print('Info inserted')

insert_info(patient1)

Arun
19
{'Dad': '23455', 'emergency': '24444'}
Info inserted


### Computed Fields
If we want to add a field in the data that is computed, i.e this field is not given by user but we create this field by using other fields


We create this using decorator "computed_field", 'property' and then write the function. Name of function will be the name of computed filed created

In [19]:
from pydantic import computed_field

class Patient(BaseModel):
    name: str
    age: int
    height: Annotated[float, Field(description="Height in meters")]
    weight: Annotated[float, Field(description="Weight in kgs")]

    @computed_field
    @property
    def bmi(self) -> float:
        computed_bmi = round((self.weight) / (self.height ** 2), 2)
        return computed_bmi
    
patient1_info = {'name':'Arun', 'age':19, 'weight':81, 'height':1.81}
patient1 = Patient(**patient1_info)



def insert_info(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.weight)
    print(patient.height)
    print(patient.bmi)
    
    print('Info inserted')

insert_info(patient1)

Arun
19
81.0
1.81
24.72
Info inserted


### Nested Models 
We can directly give a pydantic model to another, this is called Nesting

Ex: lets make a model 'address' for validation of addresses, then use it for Patient model

In [32]:
class Address(BaseModel):
    city: str
    state: str
    pincode: Annotated[int, Field(ge=100000, le=999999)]

class Patient(BaseModel):
    name: str
    age: int
    address: Address   #Look

address1_info = {'city':'Andheri', 'state':'Maharashtra','pincode':112233}
address1 = Address(**address1_info)


patient1_info = {'name':'Arun', 'age':19, 'address':address1}
patient1 = Patient(**patient1_info)


def insert_info(patient: Patient):
    print(patient.name)
    print(patient.age)
    print(patient.address)
    print(patient.address.city)
    print(patient.address.state)
    print(patient.address.pincode)
    
    print('Info inserted')

insert_info(patient1)

Arun
19
city='Andheri' state='Maharashtra' pincode=112233
Andheri
Maharashtra
112233
Info inserted


### Serialization
Serialization is the process of converting a live python object into transferable format such as JSON, Dictionary so that the object can be stored,shared and transmitted

In [33]:
#Object
patient1

Patient(name='Arun', age=19, address=Address(city='Andheri', state='Maharashtra', pincode=112233))

In [37]:
#Storing as dict
P1 = patient1.model_dump()
print(P1)
print(type(P1))

{'name': 'Arun', 'age': 19, 'address': {'city': 'Andheri', 'state': 'Maharashtra', 'pincode': 112233}}
<class 'dict'>


In [ ]:
#Storing as JSON
P1 = patient1.model_dump_json()
print(P1)
print(type(P1))
#Dumped as string but can be extracted as json

{"name":"Arun","age":19,"address":{"city":"Andheri","state":"Maharashtra","pincode":112233}}
<class 'str'>


Pydantic provides some flexibility in this too

In [40]:
# Include
P1 = patient1.model_dump(include=['name','age']) 
print(P1)


{'name': 'Arun', 'age': 19}


In [47]:
# Exclude
P1 = patient1.model_dump(exclude=['name','age'])
print(P1)

#OR for nested models
P1 = patient1.model_dump(exclude={'name':True,'age':True, 'address':['state']})
print(P1)

{'address': {'city': 'Andheri', 'state': 'Maharashtra', 'pincode': 112233}}
{'address': {'city': 'Andheri', 'pincode': 112233}}
